# 🌳 Manacá Lab

### Experimentação com IA Brasileira

Este notebook permite testar o **Manacá-1B**, um modelo de linguagem em português brasileiro, por meio de uma interface gráfica com **Gradio**.

### Como usar
1. No menu do Colab, vá em **Ambiente de execução → Alterar tipo de ambiente de execução → GPU**.
2. Clique em **Ambiente de execução → Executar tudo**.
3. Aguarde o carregamento do modelo.
4. Ao final, use a interface que aparecerá abaixo da última célula.

> **Observação:** o Manacá-1B é um modelo base. Ele funciona melhor quando recebe o início de uma frase ou parágrafo para continuar o texto.

In [ ]:
!pip install -q transformers accelerate sentencepiece gradio


## 1. Verificar o ambiente

Esta célula verifica se o Google Colab disponibilizou uma GPU.

In [ ]:
import torch

print('GPU disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Aviso: o modelo pode ficar mais lento sem GPU.')


## 2. Carregar o Manacá-1B

Na primeira execução, o modelo será baixado automaticamente.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODELO_ID = 'menezesbruno/manaca-1b-base'

tokenizer = AutoTokenizer.from_pretrained(MODELO_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODELO_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto'
)

print('✅ Manacá-1B carregado com sucesso!')


## 3. Interface do Manacá Lab

Use a interface abaixo para gerar continuações de texto.

In [ ]:
import gradio as gr

def gerar_texto(prompt, max_tokens, temperatura):
    if not prompt or not prompt.strip():
        return 'Digite um texto para começar.'

    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=int(max_tokens),
            do_sample=True,
            temperature=float(temperatura),
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

with gr.Blocks(title='Manacá Lab') as app:
    gr.Markdown('''
    # 🌳 Manacá Lab
    ### Experimentação com IA Brasileira

    Interface experimental para exploração do **Manacá-1B**.

    **Dica:** escreva o início de uma frase ou parágrafo e deixe o modelo continuar.
    ''')

    with gr.Row():
        with gr.Column():
            prompt = gr.Textbox(
                label='Início do texto',
                placeholder='Ex.: A inteligência artificial aplicada ao Direito pode...',
                lines=8
            )

            max_tokens = gr.Slider(
                minimum=50,
                maximum=500,
                value=150,
                step=10,
                label='Tamanho da geração'
            )

            temperatura = gr.Slider(
                minimum=0.1,
                maximum=1.5,
                value=0.7,
                step=0.1,
                label='Criatividade'
            )

            gerar = gr.Button('Gerar texto', variant='primary')

        with gr.Column():
            resultado = gr.Textbox(
                label='Texto gerado pelo Manacá',
                lines=16
            )

    gr.Examples(
        examples=[
            ['A inteligência artificial aplicada à recuperação da informação jurídica pode'],
            ['No contexto das bibliotecas universitárias, a inteligência artificial pode auxiliar'],
            ['A utilização de inteligência artificial no Poder Judiciário brasileiro apresenta'],
            ['A recuperação da informação jurídica consiste em']
        ],
        inputs=prompt
    )

    gerar.click(
        fn=gerar_texto,
        inputs=[prompt, max_tokens, temperatura],
        outputs=resultado
    )

app.launch(share=True)


## Sobre o experimento

Este notebook foi organizado para demonstração e experimentação acadêmica com modelos de linguagem brasileiros.

### Possíveis evoluções
- integração com documentos em PDF;
- RAG para recuperação da informação;
- experimentos com informação jurídica;
- comparação com outros modelos;
- avaliação de respostas e alucinações.

### Referências
- Repositório do Manacá-1B: https://github.com/Instituto-IA-LNCC/manaca-1b-base
- Modelo no Hugging Face: https://huggingface.co/menezesbruno/manaca-1b-base
